In [1]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 100)

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

In [5]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [7]:
from PyDI.entitymatching import EmbeddingBlocker

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks_sample, amazon_sample,
    text_cols=['title', 'author'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks_sample, goodreads_sample,
    text_cols=['title', 'author'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

embedding_candidates_m2a = embedding_blocker_m2a.materialize()
embedding_candidates_m2g = embedding_blocker_m2g.materialize()

In [72]:
from PyDI.entitymatching import EntityMatchingEvaluator


# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2a,
    blocker=embedding_blocker_m2a,
    test_pairs=train_m2a,
    out_dir=BLOCK_EVAL_DIR
)


# evaluate blocking: metabooks -> amazon
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2g,
    blocker=embedding_blocker_m2g,
    test_pairs=train_m2g,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2a)
display(results_m2g)

{'pair_completeness': 0.9945652173913043,
 'pair_quality': 0.00026165321940694794,
 'reduction_ratio': 0.9994290620408163,
 'total_candidates': 699399,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 183,
 'total_true_pairs': 184,
 'evaluation_timestamp': '2025-11-12T18:53:54.440750',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

{'pair_completeness': 0.9875,
 'pair_quality': 0.0002265703981472851,
 'reduction_ratio': 0.9994307306122449,
 'total_candidates': 697355,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 158,
 'total_true_pairs': 160,
 'evaluation_timestamp': '2025-11-12T18:54:13.139511',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [92]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    
    StringComparator(column='author',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='jaro_winkler', preprocess=str.lower),
    

    NumericComparator(column='publish_year'),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower, 
                     list_strategy='concatenate'),

    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="numratings", max_difference=100)
]

In [93]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor = FeatureExtractor(comparators)

train_features_m2a = feature_extractor.create_features(
    metabooks_sample, amazon_sample, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks_sample, goodreads_sample, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [94]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200,500],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.1, 1, 10],
        'penalty': ['l1','l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...


/Users/onurcanmemis/Desktop/Lectures/2025-2026_Winter/Team_Project/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
15 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/onurcanmemis/Desktop/Lectures/2025-2026_Winter/Team_Project/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/onurcanmemis/Desktop/Lectures/2025-2026_Winter/Team_Project/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **

Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


/Users/onurcanmemis/Desktop/Lectures/2025-2026_Winter/Team_Project/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
15 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/onurcanmemis/Desktop/Lectures/2025-2026_Winter/Team_Project/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/onurcanmemis/Desktop/Lectures/2025-2026_Winter/Team_Project/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **

In [95]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a = ml_matcher.match(
    metabooks_sample, amazon_sample,
    candidates=embedding_blocker_m2a,
    use_probabilities=True,
    threshold=0.7,
    id_column='id',
    trained_classifier=best_models[0],
)

correspondences_m2g = ml_matcher.match(
    metabooks_sample, goodreads_sample,
    candidates=embedding_blocker_m2g,
    use_probabilities=True,
    threshold=0.8,
    id_column='id',
    trained_classifier=best_models[1]
)

In [85]:
mask = (correspondences_m2g["score"] > 0.8) & (correspondences_m2g["score"] <= 0.85)
correspondences_m2g.loc[mask].sample(10)


,id1,id2,score,notes
4224,metabooks_355863,goodreads_7738,0.830236,ml_classifier=LogisticRegression
1923,metabooks_161199,goodreads_14224,0.843288,ml_classifier=LogisticRegression
1765,metabooks_146345,goodreads_8185,0.827506,ml_classifier=LogisticRegression
780,metabooks_64505,goodreads_17988,0.827506,ml_classifier=LogisticRegression
4322,metabooks_365769,goodreads_9259,0.834058,ml_classifier=LogisticRegression
42,metabooks_2989,goodreads_52129,0.843942,ml_classifier=LogisticRegression
1110,metabooks_93909,goodreads_5479,0.832462,ml_classifier=LogisticRegression
2066,metabooks_173927,goodreads_51471,0.810719,ml_classifier=LogisticRegression
991,metabooks_85227,goodreads_42340,0.840667,ml_classifier=LogisticRegression
1655,metabooks_136968,goodreads_51962,0.829325,ml_classifier=LogisticRegression


In [91]:
display(metabooks_sample[metabooks_sample.id=="metabooks_85227"])
display(goodreads_sample[goodreads_sample.id=="goodreads_42340"])

,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
5250,metabooks_85227,return of the indian the,lynne reid banks,4.7,1271,English,"Children's Books, Science Fiction, Fantasy",None,None,208,HarperColl,<NA>,6.21,0380702843


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
31298,goodreads_42340,the return of the indian,lynne reid banks,3.85,11666,English,"Fantasy, Fiction, Childrens, Young Adult, Juvenile, Adventure, Historical Fiction, Middle Grade,...",Hardcover,First Edition,183,"Doubleday & Company, Inc.",1986,3.87,038523497X


In [97]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,18924,87.324074
1,3,914,4.217618
2,4,1147,5.292788
3,5,204,0.941350
4,6,178,0.821374
...,...,...,...
30,50,1,0.004614
31,57,1,0.004614
32,69,1,0.004614
33,71,1,0.004614



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5120,91.363312
1,3,400,7.137759
2,4,55,0.981442
3,5,12,0.214133
4,6,7,0.124911
5,7,3,0.053533
6,8,1,0.017844
7,9,3,0.053533
8,15,1,0.017844
9,16,2,0.035689


In [98]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a = clusterer.cluster(correspondences_m2a)
mbm_correspondences_m2g = clusterer.cluster(correspondences_m2g)
all_correspondences = pd.concat([mbm_correspondences_m2a, mbm_correspondences_m2g], ignore_index=True)
len(all_correspondences)

30289

In [99]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

metabooks_sample.attrs["trust_score"] = 3
goodreads_sample.attrs["trust_score"] = 2
amazon_sample.attrs["trust_score"] = 1
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('language', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('numratings', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres',union)

engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_embedding_blocker.jsonl")

fused_ml_emblocker = engine.run(
    datasets=[amazon_sample, metabooks_sample, goodreads_sample],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False
)
fused_ml_emblocker.to_parquet(PIPELINE_DIR / "fused_ml_emblocker.parquet")
print(f'Fused rows: {len(fused_ml_emblocker):,}')

Fused rows: 27,990


In [100]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd


def categories_set_equal(a, b) -> bool:
    """Return True if a and b contain the same unique categories (order/type agnostic)."""
    def to_set(x):
        def items(v):
            # missing
            if v is None or (isinstance(v, float) and np.isnan(v)): return []
            # numpy array → recurse over elements
            if isinstance(v, np.ndarray): 
                out=[]; [out.extend(items(e)) for e in v.flatten()]; return out
            # python containers → recurse over elements
            if isinstance(v, (list, tuple, set)):
                out=[]; [out.extend(items(e)) for e in v]; return out
            # scalar/string: try parse stringified list; else split by delimiters
            s = str(v).strip()
            if s == "" or s.lower() in {"nan","none"}: return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)): return items(parsed)
            except Exception:
                pass
            return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]
        return {it.lower() for it in items(x)}
    return to_set(a) == to_set(b)

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("language", tokenized_match)
strategy.add_evaluation_function("genres", categories_set_equal)

fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_emblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")

In [103]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.678
  macro_accuracy: 0.678
  num_evaluated_records: 38828
  num_evaluated_attributes: 9
  total_evaluations: 342014
  total_correct: 231813
  page_count_accuracy: 0.674
  page_count_count: 35411
  publish_year_accuracy: 0.677
  publish_year_count: 38708
  title_accuracy: 0.687
  title_count: 38828
  numratings_accuracy: 0.687
  numratings_count: 38828
  language_accuracy: 0.703
  language_count: 38695
  genres_accuracy: 0.657
  genres_count: 37448
  author_accuracy: 0.652
  author_count: 38828
  price_accuracy: 0.674
  price_count: 36440
  publisher_accuracy: 0.688
  publisher_count: 38828

Overall Accuracy: 67.8%


In [22]:
mask = fused_dataset["_fusion_sources"].apply(
    lambda sources: isinstance(sources, (list, tuple, np.ndarray))
    and "amazon_sample" in sources
)

rows_from_amazon = fused_dataset[mask]


In [24]:
metabooks_sample[metabooks_sample["isbn_clean"] == "034525483X"]

,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
6279,metabooks_102449,the princess bride,william goldman,4.7,15444,English,"Science Fiction, Fantasy",None,None,<NA>,Ballantine Books,<NA>,69.620003,034525483X


In [29]:
golden = golden_fused_dataset[["isbn_clean", "title"]].rename(
    columns={"title": "title_golden"}
)
fused = fused_dataset[["isbn_clean", "title"]].rename(
    columns={"title": "title_fused"}
)

merged = golden.merge(fused, on="isbn_clean", how="inner")
diff = merged[merged["title_golden"] != merged["title_fused"]]

diff[["isbn_clean", "title_golden","title_fused"]].head()


,isbn_clean,title_golden,title_fused
4,0002160595,c s lewis a biography,jack a life of c s lewis
125,0020259913,the empire of the atom collier nucleus science fiction classic,the course of empire
128,0020292651,be expert with map and compass the complete orienteering handbook,be expert with map and compass the complete orienteering handbook emblem editions
131,002029865X,the steps of the sun,dark sun the making of the hydrogen bomb
159,0023895306,nicomachean ethics aristotle,nicomachean ethics


In [37]:
datasets = [
    ("amazon", amazon_sample),
    ("goodreads", goodreads_sample),
    ("metabooks", metabooks_sample),
    ("fused", fused_dataset),
    ("golden", golden_fused_dataset),
]

def find_title(term: str, column: str = "title"):
    for name, df in datasets:
        hits = df[df[column].str.contains(term, case=False, na=False)]
        if not hits.empty:
            print(f"=== {name} ({len(hits)} matches) ===")
            display(hits.head())  # or print(hits.head())

# usage
find_title("0684142708","isbn_clean")


=== amazon (1 matches) ===


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
17532,amazon_155957,be expert with map and compass the complete orienteering handbook emblem editions,bjorn kjellstrom,None,None,None,None,None,None,None,Macmillan General Reference,1976,None,0684142708


=== metabooks (1 matches) ===


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
9065,metabooks_147691,be expert with map compass the complete orienteering handbook emblem editions,bjorn kjellstrom,4.1,26,English,,None,None,214,Charles Scribner's Sons,<NA>,14.39,0684142708


=== golden (1 matches) ===


,isbn_clean,title,author,publisher,publish_year,language,numratings,genres,page_count,price
23653,0684142708,be expert with map and compass the complete orienteering handbook emblem editions,bjorn kjellstrom,Charles Scribner's Sons,1976,English,26,None,214,14.39
